# Adaptive Memory — EDA & Evaluation Starter

This notebook is where the data cleaning, visualization, and benchmark
comparison work happens (the actual "EDA" deliverable for the review).

Sections:
1. Load a benchmark dataset (LoCoMo / LongMemEval / a custom conversation set)
2. Clean it (dedupe, normalize, handle missing fields)
3. Run it through the memory pipeline
4. Compare against truncation and single-summary baselines
5. Visualize results


In [ ]:
import sys
sys.path.append("..")  # so `memory_engine` imports work from evaluation/

import json
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

from memory_engine.pipeline import process_message, get_store
from memory_engine.retrieve import retrieve_relevant


## 1. Load a dataset

Download LoCoMo / LongMemEval / ConvoMem into `../data/` and point this at
the actual file. For now this loads a tiny placeholder conversation so the
rest of the notebook runs end-to-end without external data.


In [ ]:
# TODO: replace with the real benchmark file, e.g.:
# with open("../data/locomo10.json") as f:
#     raw = json.load(f)

raw_conversations = [
    {
        "conversation_id": "demo_1",
        "turns": [
            {"role": "user", "text": "I live in Delhi and I work as a data analyst."},
            {"role": "user", "text": "My favorite food is biryani."},
            {"role": "user", "text": "I'm trying to learn Spanish this year."},
            {"role": "user", "text": "Actually I just moved to Mumbai last week."},
            {"role": "user", "text": "I finished the Spanish beginner course already."},
        ],
    }
]

df_raw = pd.json_normalize(
    raw_conversations, record_path="turns", meta=["conversation_id"]
)
df_raw.head()


## 2. Clean the data

Typical issues in raw conversation logs: duplicate/near-duplicate turns,
empty strings, inconsistent whitespace, missing role labels. Adjust these
checks once you're working with the real benchmark files — their schemas
differ from this placeholder.


In [ ]:
def clean_turns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["text"] = df["text"].str.strip()
    df = df[df["text"].str.len() > 0]
    df = df.drop_duplicates(subset=["conversation_id", "text"])
    df["role"] = df["role"].fillna("user")
    return df.reset_index(drop=True)

df_clean = clean_turns(df_raw)
print(f"Rows before cleaning: {len(df_raw)}, after: {len(df_clean)}")
df_clean


## 3. Run the memory pipeline on each turn

This calls your real `memory_engine` pipeline (categorize -> score ->
contradiction check -> store) on every cleaned turn, and logs what happened
to each one.


In [ ]:
results = []
for _, row in df_clean.iterrows():
    result = process_message(row["conversation_id"], row["text"])
    results.append({**result, "text": row["text"], "conversation_id": row["conversation_id"]})
    time.sleep(0.2)  # be gentle on the free-tier rate limit

df_results = pd.DataFrame(results)
df_results


## 4. Compare against baselines

Placeholder baseline implementations — swap in real ones (simple truncation
keeping last N turns; a single LLM-generated summary) so the comparison
means something for your report. The metric functions below are stubs to
fill in once you're running the real benchmarks.


In [ ]:
def truncation_baseline(turns: list[str], keep_last: int = 3) -> list[str]:
    return turns[-keep_last:]

def token_estimate(text: str) -> int:
    # Rough approximation — swap for a real tokenizer (e.g. tiktoken) once
    # you care about exact counts for the report.
    return len(text.split())

# Example: token usage comparison for one conversation
conv_turns = df_clean[df_clean["conversation_id"] == "demo_1"]["text"].tolist()

full_context_tokens = sum(token_estimate(t) for t in conv_turns)
truncated_tokens = sum(token_estimate(t) for t in truncation_baseline(conv_turns))

adaptive_items = get_store().get_all("demo_1")
adaptive_tokens = sum(token_estimate(i["text"]) for i in adaptive_items)

comparison = pd.DataFrame({
    "method": ["full_context", "truncation", "adaptive_memory"],
    "token_usage": [full_context_tokens, truncated_tokens, adaptive_tokens],
})
comparison


## 5. Visualize

Starter charts — the ones worth having in the report are the category
distribution, the token-usage comparison, and (once you have real benchmark
runs) accuracy/consistency vs. conversation length.


In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
sns.barplot(data=comparison, x="method", y="token_usage", ax=ax)
ax.set_title("Token usage by memory method")
ax.set_ylabel("Estimated tokens")
plt.tight_layout()
plt.savefig("../results_token_usage.png", dpi=150)  # create a results/ folder for these once underway
plt.show()


In [ ]:
category_counts = df_results["category"].value_counts().reset_index()
category_counts.columns = ["category", "count"]

fig, ax = plt.subplots(figsize=(6, 4))
sns.barplot(data=category_counts, x="category", y="count", ax=ax)
ax.set_title("Memory category distribution")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()


## Next steps

- Swap the placeholder conversation for real LoCoMo / LongMemEval data
- Implement the single-summary baseline (one LLM call condensing the whole
  history) alongside truncation
- Build the 20-30 conversation hand-crafted contradiction test set and run
  precision/recall on `memory_engine.contradiction`
- Track consistency/factual-accuracy scores as conversation length grows —
  this is the chart that shows your system's value over the baselines
